In [1]:
import requests
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

def carregar_dados_api_em_csv(geo_codigo, doenca,
                              semana_epidem_inicio,
                              semana_epidem_fim,
                              ano_epidem_inicio,
                              ano_epidem_fim):
  url_base = "https://info.dengue.mat.br/api/alertcity"
  parametros_requisicao = {
    "geocode": geo_codigo,
    "disease": doenca,
    "format": "csv",
    "ew_start": semana_epidem_inicio,
    "ew_end": semana_epidem_fim,
    "ey_start": ano_epidem_inicio,
    "ey_end": ano_epidem_fim}

  resposta = requests.get(url_base, params=parametros_requisicao)
  if(resposta.status_code != 200):
    print("Erro na requisição...")

  with open(f"{doenca}.csv", "w") as f:
    f.write(resposta.text)

carregar_dados_api_em_csv(2928703, "dengue", 1, 53, 2000, 2026)
carregar_dados_api_em_csv(2928703, "chikungunya", 1, 53, 2000, 2026)
carregar_dados_api_em_csv(2928703, "zika", 1, 53, 2000, 2026)

In [11]:
dados_dengue = pd.read_csv("dengue.csv")
dados_chikungunya = pd.read_csv("chikungunya.csv")
dados_zika = pd.read_csv("zika.csv")

In [12]:
colunas_inuteis = ['pop','p_inc100k','notif_accum_year','tempmax','nivel_inc','tempmin','umidmin','umidmax','transmissao','receptivo','Rt','nivel','p_rt1','casos_est_max','casos_est_min','casos_est','SE','Localidade_id', 'id', 'versao_modelo', 'municipio_nome', 'tweet', 'casprov', 'casprov_est', 'casprov_est_min', 'casprov_est_max', 'casconf']

In [13]:
dados_dengue = dados_dengue.drop(columns=colunas_inuteis)
dados_chikungunya = dados_chikungunya.drop(columns=colunas_inuteis)
dados_zika = dados_zika.drop(columns=colunas_inuteis)

In [15]:
dados_dengue

,data_iniSE,casos,umidmed,tempmed
0,2026-08-16,1,0.000000,0.000000
1,2026-08-09,10,0.000000,0.000000
2,2026-08-02,26,73.135717,17.671533
3,2026-07-26,40,89.650200,21.113829
4,2026-07-19,42,89.699886,20.946471
...,...,...,...,...
863,2010-01-31,0,74.697437,28.655242
864,2010-01-24,0,76.221839,28.148011
865,2010-01-17,0,80.623241,27.698323
866,2010-01-10,0,82.638528,27.503644


In [16]:
def dados_nulos(df, coluna):
  df[coluna].fillna(df[coluna].mean(), inplace=True)

for coluna in ["umidmed","tempmed"]:
  dados_nulos(dados_dengue, coluna)
  dados_nulos(dados_chikungunya, coluna)
  dados_nulos(dados_zika, coluna)

In [18]:
def duplicados_ordenacao_data(df):
  df.drop_duplicates(inplace=True)
  df.sort_values("data_iniSE", inplace=True)
  df["data_iniSE"] = pd.to_datetime(df["data_iniSE"])

duplicados_ordenacao_data(dados_dengue)
duplicados_ordenacao_data(dados_chikungunya)
duplicados_ordenacao_data(dados_zika)

In [19]:
def salvar_dataset(df, nome):
  caminho = "./dados"
  df.to_csv(f"{caminho}/{nome}_tratado.csv", index=False)

salvar_dataset(dados_dengue, "dengue")
salvar_dataset(dados_chikungunya, "chikungunya")
salvar_dataset(dados_zika, "zika")